# Web Scraping

## Billboard Scraping

The first step is to scrape the Billboard Global 200 chart for a given week.

In [299]:
import time
from operator import contains

#Import packages
import requests
import lxml.html as lx
import re
import pandas as pd

In [300]:
#Set up base url and headers for scraping
base_url = 'https://www.billboard.com/charts'
headers = {
    'User-Agent': "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.0.1 Safari/605.1.15"
}

In [301]:
def clean_string(stc):
    new_string = re.findall(r'(\S.*\S)', stc)[0]
    return new_string

In [302]:
def clean_int(stc):
    new_int = re.sub(r'\s*', '', stc)
    return new_int

In [303]:
def get_billboard_data(billboard_html, row):

    billboard_row = {}
    billboard_row['rank'] = row
    billboard_row['song_name'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//h3/text()")[0])

    len_artist = len(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a"))
    if len_artist == 0:
        billboard_row['artist'] = clean_string(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//li[@class='o-chart-results-list__item // lrv-u-flex-grow-1 lrv-u-flex lrv-u-flex-direction-column lrv-u-justify-content-center lrv-u-padding-l-050 lrv-u-padding-l-00@mobile-max u-max-width-397']//span/text()")[0])
        billboard_row['artist_link'] = None
    else:
        artists = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[0]
        links = billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[0]
        for i in range(1, len_artist):
            artists = artists + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/text()")[i]
            links = links + ', ' + billboard_html.xpath(f"//ul[@data-detail-target='{row}']//a/@href")[i]
        billboard_row['artist'] = artists
        billboard_row['artist_link'] = links
    billboard_row['last_week_rank'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[0].text_content())
    billboard_row['peak'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[1].text_content())
    billboard_row['weeks_chart'] = clean_int(billboard_html.xpath(f"//ul[@data-detail-target='{row}']//div[@class='lrv-u-flex lrv-u-justify-content-space-between lrv-u-align-items-center u-grid-gap-125 u-align-self-stretch u-margin-b-0.375']//li//span")[2].text_content())

    row_df = pd.DataFrame(billboard_row, index = [billboard_row["rank"]])
    return row_df

In [304]:
def get_html(link):
    billboard_info = requests.get(link, headers = headers)
    billboard_info.raise_for_status()
    billboard_html = lx.fromstring(billboard_info.text)
    return billboard_html

In [305]:
def get_billboard_chart(link):

    billboard_rows = []

    billboard_html = get_html(link)
    num_of_elements = len(billboard_html.xpath("//div[@class='o-chart-results-list-row-container']"))

    for i in range(1, num_of_elements + 1):
        new_row = get_billboard_data(billboard_html, i)
        billboard_rows.append(new_row)

    complete_chart = pd.concat(billboard_rows)
    complete_chart_indexed = complete_chart.set_index("rank", drop=True)

    return complete_chart_indexed

In [306]:
def get_billboard_chart_links(link):
    billboard_html = get_html(link)
    base_direc = 'https://www.billboard.com'

    charts_dic = {}
    chart_categories = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/text()")
    chart_links = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-hits-of-the-world']//a/@href")

    #US table
    hot_100_link = billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-top-charts']//a/@href")[0]

    #Global 200
    hot_200_link =billboard_html.xpath("//ul[@class='lrv-a-unstyle-list lrv-u-background-color-black lrv-u-color-white u-padding-b-3.125']//ul[@aria-labelledby='charts-menu-item-global']//a/@href")[0]

    for element, element_link in zip(chart_categories, chart_links):
        charts_dic[clean_string(element)] = base_direc + element_link
    charts_dic['U.S. Hot 100 Songs'] = base_direc + hot_100_link
    charts_dic['Worldwide Top 200 Songs'] = base_direc + hot_200_link

    return charts_dic

In [307]:
def get_all_billboard_global_charts(url):
    world_chart_links = get_billboard_chart_links(url)
    adjusted_world_charts = [chart_name for chart_name in list(world_chart_links.keys()) if re.search("Artist|Album|Global|Top Philippine|Thai Country|Vietnamese|Singles", chart_name) is None]

    all_charts = {}

    for chart in adjusted_world_charts:
        time.sleep(1)
        print(chart)
        chart_link = world_chart_links[chart]
        chart_table = get_billboard_chart(chart_link)
        column_add = [chart] * chart_table.shape[0]
        chart_table.insert(loc = chart_table.shape[1], column = "Chart", value = column_add)
        all_charts[chart] = chart_table

    return all_charts

In [308]:
all_charts_df = get_all_billboard_global_charts(base_url)

Billboard Arabic Hot 100
Billboard Argentina Hot 100
Billboard Brasil Hot 100
Billboard Colombia Hot 100
Billboard Canadian Hot 100
Billboard Italy Hot 100
Billboard Japan Hot 100
Billboard Korea Hot 100
Billboard Philippines Hot 100
Billboard Thailand Top Thai Songs
Billboard Vietnam Hot 100
Australia Songs
Austria Songs
Belgium Songs
Bolivia Songs
Chile Songs
China TME UNI Chart
Croatia Songs
Czech Republic Songs
Denmark Songs
Ecuador Songs
Finland Songs
France Songs
Germany Songs
Greece Songs
Hong Kong Songs
Hungary Songs
Iceland Songs
India Songs
Indonesia Songs
Ireland Songs
Luxembourg Songs
Malaysia Songs
Mexico Songs
Netherlands Songs
New Zealand Songs
Norway Songs
Peru Songs
Poland Songs
Portugal Songs
Romania Songs
Singapore Songs
Slovakia Songs
South Africa Songs
Spain Songs
Sweden Songs
Switzerland Songs
Taiwan Songs
Turkey Songs
U.K. Songs
U.S. Hot 100 Songs
Worldwide Top 200 Songs


In [309]:
all_charts_df_together = pd.concat(all_charts_df)

In [316]:
all_charts_df_together

song_name  \
                         rank                                
Billboard Arabic Hot 100 1                           FA9LA   
                         2                            Yama   
                         3                       Ma Tnsani   
                         4                           Ta3al   
                         5                  Batmanna Ansak   
...                                                    ...   
Worldwide Top 200 Songs  196                        Closer   
                         197              No Tiene Sentido   
                         198          Don't Stop The Music   
                         199                         Taste   
                         200   Have You Ever Seen The Rain   

                                                     artist  \
                         rank                                 
Billboard Arabic Hot 100 1                      Flipperachi   
                         2                         DYSTINCT   
                         3              Vanco Featuring AYA   
                         4                         DYSTINCT   
                         5              Sherine Abdel Wahab   
...                                                     ...   
Worldwide Top 200 Songs  196       The Chainsmokers, Halsey   
                         197                          Beele   
                         198                        Rihanna   
                         199              Sabrina Carpenter   
                         200   Creedence Clearwater Revival   

                                                                     artist_link  \
                         rank                                                      
Billboard Arabic Hot 100 1                                                  None   
                         2                                                  None   
                         3                                                  None   
                         4                                                  None   
                         5                                                  None   
...                                                                          ...   
Worldwide Top 200 Songs  196   https://www.billboard.com/artist/the-chainsmok...   
                         197                                                None   
                         198           https://www.billboard.com/artist/rihanna/   
                         199   https://www.billboard.com/artist/sabrina-carpe...   
                         200   https://www.billboard.com/artist/creedence-cle...   

                              last_week_rank peak weeks_chart  \
                         rank                                   
Billboard Arabic Hot 100 1                 1    1           9   
                         2                 2    1          16   
                         3                 3    2          28   
                         4                 4    3           3   
                         5                 5    5          34   
...                                      ...  ...         ...   
Worldwide Top 200 Songs  196             150   96         142   
                         197             167   27          33   
                         198             189  138           3   
                         199               -    2          66   
                         200             166   92          74   

                                                  Chart  
                         rank                            
Billboard Arabic Hot 100 1     Billboard Arabic Hot 100  
                         2     Billboard Arabic Hot 100  
                         3     Billboard Arabic Hot 100  
                         4     Billboard Arabic Hot 100  
                         5     Billboard Arabic Hot 100  
...                                                 ...  
Worldwide Top 200 S

In [311]:
all_charts_df_together.groupby("artist").count().sort_values(by=['song_name'], ascending = False).iloc[:, 0]

artist
Bad Bunny                                        205
Olivia Dean                                       61
Taylor Swift                                      46
J. Cole                                           46
sombr                                             37
                                                ... 
J. Cole, Erykah Badu                               1
J. Cole, Burna Boy                                 1
Ivan Greko, Michalis Karagkouni & Sin Laurent      1
Iuliana Beregoi & Cristian Porcari                 1
wtfrank                                            1
Name: song_name, Length: 658, dtype: int64

In [356]:
all_charts_df_together.groupby(by = ["artist", "Chart"]).artist.agg("count")

artist                 Chart                     
.Feast                 Indonesia Songs               2
1119                   Malaysia Songs                1
14 Casper, Bon Nghiem  Billboard Vietnam Hot 100     1
3 Doors Down           Billboard Canadian Hot 100    1
                       Worldwide Top 200 Songs       2
                                                    ..
sombr                  Switzerland Songs             1
                       U.K. Songs                    3
                       U.S. Hot 100 Songs            3
                       Worldwide Top 200 Songs       4
wtfrank                Turkey Songs                  1
Name: artist, Length: 1179, dtype: int64

In [338]:
temp = all_charts_df_together.groupby(by = ["artist", "Chart"]).groupby("artist").count().sort_values(by=['song_name'], ascending = False).iloc[:, 0]

AttributeError: 'DataFrameGroupBy' object has no attribute 'groupby'

In [324]:
temp.groupby("artist").count().sort_values(by=['song_name'], ascending = False).iloc[:, 0]

AttributeError: 'DataFrameGroupBy' object has no attribute 'groupby'